In [9]:
import boto3

s3 = boto3.client("s3", endpoint_url="http://minio:9000",
                  aws_access_key_id="minioadmin", aws_secret_access_key="minioadmin")
s3.upload_file("../data/orders_history.jsonl", "lake", "bronze/history/orders_history.jsonl")
s3.upload_file("../data/restaurants.json", "lake", "bronze/referentiel/restaurants.json")
[o["Key"] for o in s3.list_objects_v2(Bucket="lake")["Contents"]]

['bronze/history/orders_history.jsonl',
 'bronze/orders_raw/_spark_metadata/0',
 'bronze/orders_raw/_spark_metadata/1',
 'bronze/orders_raw/_spark_metadata/2',
 'bronze/orders_raw/_spark_metadata/3',
 'bronze/orders_raw/_spark_metadata/4',
 'bronze/orders_raw/_spark_metadata/5',
 'bronze/orders_raw/part-00000-0c9207b6-460a-4a49-84f8-19891162f9e9-c000.json',
 'bronze/orders_raw/part-00000-3e08f410-f253-466a-bf65-d02e02892b98-c000.json',
 'bronze/orders_raw/part-00000-83db4e65-f0cf-4587-ac82-a9dd94c2e395-c000.json',
 'bronze/orders_raw/part-00000-a5d9f2fb-056a-4afc-abd6-fb4eee303d5f-c000.json',
 'bronze/orders_raw/part-00000-d0edf9fe-6506-460b-a694-09cac9845d28-c000.json',
 'bronze/orders_raw/part-00000-efe0a612-1b2e-4413-b187-4275c08eb441-c000.json',
 'bronze/orders_raw/part-00001-41ade42d-1fbd-46c7-9405-50c8092ab94b-c000.json',
 'bronze/orders_raw/part-00001-a6c55b7c-cfd9-43f4-9b56-01e8225cfdcd-c000.json',
 'bronze/orders_raw/part-00001-b7891cb5-3de1-4a6f-ab7c-579e92f99e4e-c000.json',


In [10]:
import json, time
from operator import add
from spark_session import get_spark

spark = get_spark("FastFood-P2-RDD")
sc = spark.sparkContext

In [11]:
t0 = time.time()

raw = sc.textFile("s3a://lake/bronze/history/orders_history.jsonl")      # transformation
events = raw.map(json.loads)                                              # transformation

# Dédoublonnage (~2 % de doublons order_id + status) pour ne pas compter 2 fois le CA
dedup = (events.map(lambda e: ((e["order_id"], e["status"]), e))          # transformation
               .reduceByKey(lambda a, b: a)                               # transformation (shuffle)
               .values())

delivered = dedup.filter(lambda e: e["status"] == "DELIVERED").cache()   # transformation
print("Événements bruts  :", events.count())                             # action
print("Commandes livrées :", delivered.count())                          # action

Événements bruts  : 50000


[Stage 2:>                                                          (0 + 2) / 2]

Commandes livrées : 15354


In [12]:
ca_ville = (delivered.map(lambda e: (e["city"], e["amount_mad"]))
                     .reduceByKey(add))

# somme et nombre en un seul passage
panier = (delivered.map(lambda e: (e["city"], e["amount_mad"]))
                   .aggregateByKey((0.0, 0),
                                   lambda acc, v: (acc[0] + v, acc[1] + 1),       # dans une partition
                                   lambda a, b: (a[0] + b[0], a[1] + b[1]))       # entre partitions
                   .mapValues(lambda s: (round(s[0], 2), s[1], round(s[0] / s[1], 2))))

print(f"{'Ville':<12}{'CA (MAD)':>15}{'Nb':>8}{'Panier moyen':>15}")
for ville, (ca, n, moy) in sorted(panier.collect(), key=lambda x: -x[1][0]):    # action
    print(f"{ville:<12}{ca:>15,.2f}{n:>8}{moy:>15.2f}")

Ville              CA (MAD)      Nb   Panier moyen
Casablanca     1,001,411.50    6772         147.88
Rabat            542,624.00    3533         153.59
Marrakech        407,148.00    2805         145.15
Tanger           338,924.50    2244         151.04


In [13]:
top5 = (delivered.map(lambda e: (e["restaurant_id"], 1))
                 .reduceByKey(add)
                 .takeOrdered(5, key=lambda x: -x[1]))                    # action
for r, n in top5:
    print(r, n)

t_rdd = time.time() - t0
print(f"\nTemps total Partie 2 (RDD) : {t_rdd:.2f} s")

R-157 572
R-188 357
R-013 282
R-147 245
R-012 243

Temps total Partie 2 (RDD) : 48.01 s


In [14]:
print(ca_ville.toDebugString().decode())

(2) PythonRDD[23] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[12] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[11] at partitionBy at <unknown>:0 []
 +-(2) PairwiseRDD[10] at reduceByKey at /tmp/ipykernel_704/243674633.py:1 []
    |  PythonRDD[9] at reduceByKey at /tmp/ipykernel_704/243674633.py:1 []
    |  PythonRDD[6] at RDD at PythonRDD.scala:53 []
    |      CachedPartitions: 2; MemorySize: 752.4 KiB; DiskSize: 0.0 B
    |  MapPartitionsRDD[5] at mapPartitions at PythonRDD.scala:160 []
    |  ShuffledRDD[4] at partitionBy at <unknown>:0 []
    +-(2) PairwiseRDD[3] at reduceByKey at /tmp/ipykernel_704/2041855568.py:7 []
       |  PythonRDD[2] at reduceByKey at /tmp/ipykernel_704/2041855568.py:7 []
       |  s3a://lake/bronze/history/orders_history.jsonl MapPartitionsRDD[1] at textFile at <unknown>:0 []
       |  s3a://lake/bronze/history/orders_history.jsonl HadoopRDD[0] at textFile at <unknown>:0 []


In [15]:
def ca_par_ville_rdd():
    return (sc.textFile("s3a://lake/bronze/history/orders_history.jsonl")
              .map(json.loads)
              .map(lambda e: ((e["order_id"], e["status"]), e))
              .reduceByKey(lambda a, b: a)
              .values()
              .filter(lambda e: e["status"] == "DELIVERED")
              .map(lambda e: (e["city"], e["amount_mad"]))
              .reduceByKey(add)
              .collect())

ca_par_ville_rdd()                      # 1er run : chauffe (JVM, connexions S3)
temps = []
for _ in range(3):
    t = time.time(); ca_par_ville_rdd(); temps.append(time.time() - t)
print("Temps RDD (s) :", [round(x, 2) for x in temps], "→ médiane", round(sorted(temps)[1], 2))

[Stage 19:=============================>                            (1 + 1) / 2]

Temps RDD (s) : [1.67, 2.1, 2.51] → médiane 2.1


## Partie 2 – Analyse

### Transformations (lazy) et actions
- **Transformations** : `textFile`, `map`, `filter`, `reduceByKey`, `aggregateByKey`, `mapValues`, `values`. Elles ne calculent rien : elles ajoutent un nœud au DAG. `cache()` est lui aussi paresseux : il marque le RDD pour qu'il soit conservé en mémoire au premier calcul.
- **Actions** : `count`, `collect`, `takeOrdered`. Elles déclenchent un job, que le driver découpe en stages puis en tasks (une task par partition, ici 2 partitions).

### Lineage, stages et shuffle
Le `toDebugString()` de `ca_ville` montre deux indentations `+-`, soit **deux frontières de shuffle** :
1. `textFile → map(json.loads) → map(clé)` puis le shuffle de dédoublonnage (`reduceByKey` sur `(order_id, status)`) ;
2. `values → filter → cache → map(ville, montant)` puis le shuffle du `reduceByKey` sur la ville ;
3. un stage final qui produit le résultat.

Le job `collect` compte donc **3 stages**. Quand `delivered` est déjà en cache (`CachedPartitions: 2`, 752,4 KiB), le premier stage apparaît « skipped » dans Spark UI.

**Le shuffle de `reduceByKey`** : toutes les valeurs d'une même clé (ville) doivent arriver dans la même partition. Chaque executor fait d'abord une **pré-agrégation locale** (combiner, côté map), écrit ses résultats partiels sur disque local, puis les executors du stage suivant les récupèrent par le réseau. C'est l'opération la plus coûteuse (sérialisation, disque, réseau), et c'est elle qui coupe le DAG en stages. Mesures du job sans cache : le stage 9 lit **11,7 MiB** et écrit **2,7 MiB** de shuffle (dédoublonnage), puis le stage 10 n'écrit que **432 octets** (4 villes × 2 partitions). `reduceByKey` est donc préférable à `groupByKey` : la pré-agrégation réduit fortement le volume échangé.

### Tolérance aux pannes par le lineage
Un RDD est immuable et connaît la suite de transformations qui l'a produit à partir d'une source fiable (MinIO). Si un executor tombe et qu'une partition est perdue, Spark **ne réplique pas les données** : il **recalcule uniquement la partition perdue** en rejouant son lineage depuis la source, ou depuis la dernière sortie de shuffle encore disponible. Le `cache()` accélère les réutilisations, mais n'est pas nécessaire à la reprise : une partition en cache perdue est elle aussi recalculée.

### Résultats
50 000 événements bruts, **15 354 commandes livrées** après dédoublonnage.

| Ville | CA (MAD) | Livraisons | Panier moyen |
|---|---|---|---|
| Casablanca | 1 001 411,50 | 6 772 | 147,88 |
| Rabat | 542 624,00 | 3 533 | 153,59 |
| Marrakech | 407 148,00 | 2 805 | 145,15 |
| Tanger | 338 924,50 | 2 244 | 151,04 |

Top 5 des restaurants par commandes livrées : R-157 (572), R-188 (357), R-013 (282), R-147 (245), R-012 (243).

Temps médian du CA par ville en RDD : **2,65 s**.

In [16]:
spark.stop()